# DACON ETRI post-ID341 upstream rebuild runner

Report-only Kaggle runner scaffold for the post-ID341 upstream model-level rebuild blueprint. It does **not** create DACON submissions, submission CSVs, or submission-log rows. The goal is to verify the packaged project payload and freeze the no-submit rebuild contract before any heavier Kaggle job is launched.

## 1. Locate the packaged project payload

The runner accepts either a `source.zip` from the Kaggle dataset package or an unpacked project tree under `/kaggle/input`. It copies the project into `/kaggle/working/project` so any generated reports stay isolated from the read-only input dataset.

In [ ]:

import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path


def run(cmd: str, cwd: Path | None = None) -> None:
    print(f"\n$ {cmd}", flush=True)
    env = os.environ.copy()
    env.setdefault("PYTHONHASHSEED", "0")
    if cwd is not None:
        env["PYTHONPATH"] = str(cwd)
    subprocess.run(cmd, shell=True, check=True, cwd=str(cwd) if cwd else None, env=env)

is_kaggle = Path("/kaggle/input").exists()
default_input_root = "/kaggle/input" if is_kaggle else str(Path.cwd())
input_root = Path(os.environ.get("KAGGLE_INPUT_DIR", default_input_root))
working_root = Path(os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working" if is_kaggle else "build/kaggle_upstream_rebuild_smoke"))
project_dir = working_root / "project"
project_dir.parent.mkdir(parents=True, exist_ok=True)

if project_dir.exists():
    shutil.rmtree(project_dir)
project_dir.mkdir(parents=True, exist_ok=True)

COPY_NAMES = ["src", "conf", "scripts", "data", "artifacts", "submissions", "reports", "pyproject.toml", "README.md", "dataset-metadata.json", "Makefile"]

def copy_project_tree(source_root: Path, dst_root: Path) -> None:
    print(f"Overlaying project root: {source_root}")
    for name in COPY_NAMES:
        src = source_root / name
        dst = dst_root / name
        if not src.exists():
            continue
        if src.is_dir():
            shutil.copytree(src, dst, dirs_exist_ok=True, ignore=shutil.ignore_patterns("__pycache__", "*.pyc", ".DS_Store", ".ruff_cache", ".pytest_cache"))
        else:
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)

def root_priority(source_root: Path) -> tuple[int, int, str]:
    has_full_raw = (source_root / "data" / "raw" / "ch2025_data_items").exists()
    has_reports = (source_root / "reports").exists()
    return (0 if has_full_raw else 1, 0 if not has_reports else 1, str(source_root))

source_zip_candidates = sorted(input_root.rglob("source.zip")) if input_root.exists() else []
if source_zip_candidates:
    for source_zip in source_zip_candidates:
        print(f"Extracting packaged source zip overlay: {source_zip}")
        with zipfile.ZipFile(source_zip) as zf:
            zf.extractall(project_dir)

candidates = [p for p in input_root.rglob("pyproject.toml")] if input_root.exists() else []
if not candidates and not is_kaggle:
    candidates = [Path.cwd() / "pyproject.toml"]
source_roots = sorted({p.parent for p in candidates}, key=root_priority)
if not source_roots and not source_zip_candidates:
    raise FileNotFoundError("Could not locate source.zip or pyproject.toml in Kaggle inputs.")
for source_root in source_roots:
    copy_project_tree(source_root, project_dir)

metadata_path = project_dir / "dataset-metadata.json"
if not metadata_path.exists():
    metadata_path.write_text(json.dumps({
        "title": "Dacon ETRI Runtime Project Metadata",
        "id": "softkleenex/dacon-etri-upstream-runtime-overlay-post-id341",
        "licenses": [{"name": "CC0-1.0"}],
        "dataset_sources": [
            "softkleenex/dacon-etri-human-base",
            "softkleenex/dacon-etri-upstream-patch-post-id341",
        ],
        "note": "Synthesized by the no-submit runner because Kaggle consumes dataset-metadata.json as dataset metadata and may not expose it as an input file.",
    }, indent=2) + "\n", encoding="utf-8")
    print(f"Synthesized runtime metadata: {metadata_path}")

required = ["scripts", "src", "conf", "pyproject.toml", "reports", "artifacts/base_predictions", "submissions/submission_log.csv"]
missing = [name for name in required if not (project_dir / name).exists()]
print("Project dir:", project_dir)
print("Missing required paths:", missing)
if missing:
    raise FileNotFoundError(f"Packaged project is missing required paths: {missing}")



Payload verification should finish before any compute. If a required artifact or report is missing, the notebook stops here instead of producing misleading rebuild output.

## 2. Ensure runtime dependencies only if missing

Kaggle usually has the required CPU packages. This cell checks imports first and installs only absent packages.

In [ ]:

if is_kaggle:
    import importlib.util

    required_packages = {
        "numpy": "numpy",
        "pandas": "pandas",
        "sklearn": "scikit-learn",
        "yaml": "pyyaml",
        "nbformat": "nbformat",
    }
    missing_packages = [pkg for module, pkg in required_packages.items() if importlib.util.find_spec(module) is None]
    print("Missing packages:", missing_packages)
    if missing_packages:
        run(f"{sys.executable} -m pip install " + " ".join(missing_packages))
else:
    print("Local smoke mode: no package installation attempted.")


Dependency handling is intentionally conservative: no global environment mutation is attempted outside Kaggle, and the runner stays CPU-only.

## 3. Regenerate local-closure and upstream readiness reports

The runner first regenerates the latest post-ID341 local-closure reports on Kaggle CPU: transfer gate v2, external ETRI prior smoke, and the combined sleep/evening/external-prior gate. Only after those no-submit reports are refreshed does it run the upstream blueprint/readiness reports, clean-ID221 probes, clean-ID226 calibration smoke, and clean-backbone lane closure. These commands create reports only: no submission CSV, no base prediction artifact, and no submission-log row.


### Latest local-closure gates

These no-submit reports are regenerated inside the Kaggle working copy so the runner proves the current local closure under the remote CPU/runtime environment before any upstream rebuild planning.


In [ ]:
transfer_v2_path = project_dir / "reports" / "remote_post_id341_transfer_gate_v2_joint_risk_20260618.json"
cmd = f"{sys.executable} scripts/diagnose_post_id341_transfer_gate_v2_with_joint_risk.py --output {transfer_v2_path}"
run(cmd, cwd=project_dir)

transfer_v2 = json.loads(transfer_v2_path.read_text())
print(json.dumps(transfer_v2["decision"], indent=2, ensure_ascii=False))
assert transfer_v2["decision"]["any_candidate_passes_combined_public_058_gate"] is False
assert transfer_v2["decision"]["create_submission_artifact"] is False
assert transfer_v2["decision"]["submit_now"] is False


In [ ]:
external_prior_path = project_dir / "reports" / "remote_external_etri_prior_smoke_post_id341_20260618.json"
cmd = f"{sys.executable} scripts/diagnose_external_etri_prior_smoke_post_id341.py --output {external_prior_path}"
run(cmd, cwd=project_dir)

external_prior = json.loads(external_prior_path.read_text())
print(json.dumps(external_prior["decision"], indent=2, ensure_ascii=False))
assert external_prior["decision"]["external_prior_promotable_public_058_candidate_exists"] is False
assert external_prior["decision"]["create_submission_artifact"] is False
assert external_prior["decision"]["submit_now"] is False


In [ ]:
combined_external_path = project_dir / "reports" / "remote_combined_sleep_evening_external_prior_gate_post_id341_20260618.json"
cmd = f"{sys.executable} scripts/diagnose_combined_sleep_evening_external_prior_gate_post_id341.py --output {combined_external_path}"
run(cmd, cwd=project_dir)

combined_external = json.loads(combined_external_path.read_text())
print(json.dumps(combined_external["decision"], indent=2, ensure_ascii=False))
assert combined_external["decision"]["proxy_below_id178_candidate_exists"] is False
assert combined_external["decision"]["public_058_plausible"] is False
assert combined_external["decision"]["create_submission_artifact"] is False
assert combined_external["decision"]["submit_now"] is False


### Phase-2 raw/sequence source execution spec

This report freezes the next approved Kaggle revision after closure parity: a stronger raw/sequence source smoke with no submission artifacts and strict transfer/joint-risk gates.


In [ ]:
phase2_spec_path = project_dir / "reports" / "remote_post_id341_phase2_raw_source_execution_spec_20260618.json"
cmd = f"{sys.executable} scripts/diagnose_post_id341_phase2_raw_source_execution_spec.py --output {phase2_spec_path}"
run(cmd, cwd=project_dir)

phase2_spec = json.loads(phase2_spec_path.read_text())
print(json.dumps(phase2_spec["decision"], indent=2, ensure_ascii=False))
assert phase2_spec["decision"]["phase2_spec_ready"] is True
assert phase2_spec["decision"]["create_submission_artifact"] is False
assert phase2_spec["decision"]["submit_now"] is False
assert phase2_spec["decision"]["run_heavy_rebuild_now"] is False


### Phase-2 raw multimodal execution plan

This no-submit report turns the latest local weak-source evidence into a concrete Kaggle CPU block order. The first allowed block is Q1/S4 sleep-residual bridge work; it still creates no submission artifact.

In [ ]:
q2q3s3_matrix_path = project_dir / "reports" / "remote_post_id341_q2q3s3_source_hypothesis_matrix_20260618.json"
cmd = f"{sys.executable} scripts/diagnose_post_id341_q2q3s3_source_hypothesis_matrix.py --output {q2q3s3_matrix_path}"
run(cmd, cwd=project_dir)

q2q3s3_matrix = json.loads(q2q3s3_matrix_path.read_text())
print(json.dumps(q2q3s3_matrix["decision"], indent=2, ensure_ascii=False))
assert q2q3s3_matrix["decision"]["q2q3s3_source_hypothesis_matrix_ready"] is True
assert q2q3s3_matrix["decision"]["all_focus_targets_missing_locally"] is True
assert q2q3s3_matrix["decision"]["all_focus_targets_have_historical_id226_signal"] is True
assert q2q3s3_matrix["decision"]["local_q2q3s3_artifact_promotable"] is False
assert q2q3s3_matrix["decision"]["create_submission_artifact"] is False
assert q2q3s3_matrix["decision"]["submit_now"] is False
assert q2q3s3_matrix["decision"]["push_kaggle_now"] is False
assert q2q3s3_matrix["decision"]["run_kaggle_now"] is False


In [ ]:
q2q3s3_fallback_chain_path = project_dir / "reports" / "remote_post_id341_q2q3s3_local_fallback_chain_contract_20260618.json"
cmd = f"{sys.executable} scripts/diagnose_post_id341_q2q3s3_local_fallback_chain_contract.py --output {q2q3s3_fallback_chain_path}"
run(cmd, cwd=project_dir)

q2q3s3_fallback_chain = json.loads(q2q3s3_fallback_chain_path.read_text())
print(json.dumps(q2q3s3_fallback_chain["decision"], indent=2, ensure_ascii=False))
assert q2q3s3_fallback_chain["decision"]["local_fallback_chain_contract_pass"] is True
assert q2q3s3_fallback_chain["decision"]["selected_local_context_targets"] == ["Q2"]
assert q2q3s3_fallback_chain["decision"]["q3_s3_still_uncovered"] is True
assert q2q3s3_fallback_chain["decision"]["local_q2_lead_context_only"] is True
assert q2q3s3_fallback_chain["decision"]["public_058_evidence"] is False
assert q2q3s3_fallback_chain["decision"]["create_submission_artifact"] is False
assert q2q3s3_fallback_chain["decision"]["submit_now"] is False
assert q2q3s3_fallback_chain["decision"]["push_kaggle_now"] is False
assert q2q3s3_fallback_chain["decision"]["run_kaggle_now"] is False


In [ ]:
q3s3_negative_evidence_path = project_dir / "reports" / "remote_post_id341_q3s3_negative_evidence_map_20260618.json"
cmd = f"{sys.executable} scripts/diagnose_post_id341_q3s3_negative_evidence_map.py --output {q3s3_negative_evidence_path}"
run(cmd, cwd=project_dir)

q3s3_negative_evidence = json.loads(q3s3_negative_evidence_path.read_text())
print(json.dumps(q3s3_negative_evidence["decision"], indent=2, ensure_ascii=False))
assert q3s3_negative_evidence["decision"]["q3s3_negative_evidence_map_passed"] is True
assert q3s3_negative_evidence["decision"]["closed_local_lane_count"] == 8
assert q3s3_negative_evidence["decision"]["all_local_q3s3_lanes_closed"] is True
assert q3s3_negative_evidence["decision"]["create_submission_artifact"] is False
assert q3s3_negative_evidence["decision"]["submit_now"] is False
assert q3s3_negative_evidence["decision"]["push_kaggle_now"] is False
assert q3s3_negative_evidence["decision"]["run_kaggle_now"] is False

q3s3_fold_nested_sibling_path = project_dir / "reports" / "remote_post_id341_q3s3_fold_nested_sibling_bridge_smoke_20260619.json"
cmd = (
    f"{sys.executable} scripts/diagnose_post_id341_q3s3_fold_nested_sibling_bridge_smoke.py "
    f"--negative-evidence-map {q3s3_negative_evidence_path} "
    f"--output {q3s3_fold_nested_sibling_path}"
)
run(cmd, cwd=project_dir)

q3s3_fold_nested_sibling = json.loads(q3s3_fold_nested_sibling_path.read_text())
print(json.dumps(q3s3_fold_nested_sibling["decision"], indent=2, ensure_ascii=False))
assert q3s3_fold_nested_sibling["decision"]["fold_nested_sibling_smoke_completed"] is True
assert q3s3_fold_nested_sibling["decision"]["selected_targets"] == []
assert q3s3_fold_nested_sibling["decision"]["public_058_evidence"] is False
assert q3s3_fold_nested_sibling["decision"]["create_submission_artifact"] is False
assert q3s3_fold_nested_sibling["decision"]["submit_now"] is False
assert q3s3_fold_nested_sibling["decision"]["push_kaggle_now"] is False
assert q3s3_fold_nested_sibling["decision"]["run_kaggle_now"] is False

q3s3_remaining_route_inventory_path = project_dir / "reports" / "remote_post_id341_q3s3_remaining_route_inventory_20260618.json"
cmd = (
    f"{sys.executable} scripts/diagnose_post_id341_q3s3_remaining_route_inventory.py "
    f"--negative-evidence-map {q3s3_negative_evidence_path} "
    f"--fold-nested-sibling-bridge {q3s3_fold_nested_sibling_path} "
    f"--output {q3s3_remaining_route_inventory_path}"
)
run(cmd, cwd=project_dir)

q3s3_remaining_route_inventory = json.loads(q3s3_remaining_route_inventory_path.read_text())
print(json.dumps(q3s3_remaining_route_inventory["decision"], indent=2, ensure_ascii=False))
assert q3s3_remaining_route_inventory["decision"]["q3s3_remaining_route_inventory_ready"] is True
assert q3s3_remaining_route_inventory["decision"]["q3_s3_still_uncovered"] is True
assert q3s3_remaining_route_inventory["decision"]["fold_nested_sibling_bridge_closed"] is True
assert q3s3_remaining_route_inventory["decision"]["local_no_approval_score_route_exists"] is False
assert q3s3_remaining_route_inventory["decision"]["only_score_relevant_open_route"] == "explicit_approved_clean_upstream_id221_to_id226"
assert q3s3_remaining_route_inventory["decision"]["public_058_evidence"] is False
assert q3s3_remaining_route_inventory["decision"]["create_submission_artifact"] is False
assert q3s3_remaining_route_inventory["decision"]["submit_now"] is False
assert q3s3_remaining_route_inventory["decision"]["push_kaggle_now"] is False
assert q3s3_remaining_route_inventory["decision"]["run_kaggle_now"] is False


In [ ]:
phase2_plan_path = project_dir / "reports" / "remote_phase2_raw_multimodal_execution_plan_post_id341_20260618.json"
cmd = f"{sys.executable} scripts/diagnose_phase2_raw_multimodal_execution_plan_post_id341.py --output {phase2_plan_path}"
run(cmd, cwd=project_dir)

phase2_plan = json.loads(phase2_plan_path.read_text())
print(json.dumps(phase2_plan["decision"], indent=2, ensure_ascii=False))
assert phase2_plan["decision"]["phase2_raw_multimodal_plan_ready"] is True
assert phase2_plan["decision"]["recommended_first_block"] == "A_q1_s4_sleep_residual_bridge"
assert phase2_plan["decision"]["create_submission_artifact"] is False
assert phase2_plan["decision"]["submit_now"] is False
assert phase2_plan["decision"]["push_kaggle_now"] is False
assert phase2_plan["decision"]["run_kaggle_now"] is False


### Phase-2 block A: Q1/S4 sleep-residual bridge

This no-submit block regenerates the first raw multimodal source smoke on Kaggle CPU. It may pass all primary local gates but still remains non-promotable unless the projected proxy falls below the actual ID178 public anchor.

In [ ]:
block_a_path = project_dir / "reports" / "remote_phase2_block_A_q1_s4_sleep_residual_bridge_20260618.json"
cmd = f"{sys.executable} scripts/diagnose_phase2_block_a_q1_s4_sleep_residual_bridge_post_id341.py --output {block_a_path}"
run(cmd, cwd=project_dir)

block_a = json.loads(block_a_path.read_text())
print(json.dumps(block_a["decision"], indent=2, ensure_ascii=False))
assert block_a["decision"]["create_submission_artifact"] is False
assert block_a["decision"]["submit_now"] is False
assert block_a["decision"]["push_kaggle_now"] is False
assert block_a["decision"]["run_kaggle_now"] is False
# Do not assert final_block_a_gate_pass: failure is a valid report-only stop condition.


### Phase-2 block A Q1 proxy gap

This report-only cell analyzes the freshly generated block-A JSON and quantifies how far the Q1-only signal is from the ID178 public anchor and the Public 0.58 review line. It trains no model and creates no submission artifacts.


In [ ]:
block_a_gap_path = project_dir / "reports" / "remote_phase2_block_A_q1_proxy_gap_20260618.json"
cmd = (
    f"{sys.executable} scripts/diagnose_phase2_block_a_q1_proxy_gap_post_id341.py "
    f"--source-report {block_a_path} --output {block_a_gap_path}"
)
run(cmd, cwd=project_dir)

block_a_gap = json.loads(block_a_gap_path.read_text())
print(json.dumps(block_a_gap["decision"], indent=2, ensure_ascii=False))
assert block_a_gap["decision"]["create_submission_artifact"] is False
assert block_a_gap["decision"]["submit_now"] is False
assert block_a_gap["decision"]["push_kaggle_now"] is False
assert block_a_gap["decision"]["run_kaggle_now"] is False
# A False sufficiency assertion is expected unless a genuinely larger independent signal appears.


### Phase-2 block B: S2 sleep/recovery closure check

This no-submit cell is a bounded closure check for the independent S2 recovery lane. It may fail, and failure is valid evidence: the runner must still create no submission artifacts and must not submit to DACON.


In [ ]:
block_b_path = project_dir / "reports" / "remote_phase2_block_B_s2_sleep_recovery_retry_20260618.json"
cmd = f"{sys.executable} scripts/diagnose_phase2_block_b_s2_sleep_recovery_retry_post_id341.py --output {block_b_path}"
run(cmd, cwd=project_dir)

block_b = json.loads(block_b_path.read_text())
print(json.dumps(block_b["decision"], indent=2, ensure_ascii=False))
assert block_b["decision"]["create_submission_artifact"] is False
assert block_b["decision"]["submit_now"] is False
assert block_b["decision"]["push_kaggle_now"] is False
assert block_b["decision"]["run_kaggle_now"] is False
# Do not assert final_block_b_gate_pass: failure closes the local S2 recovery lane.


### Phase-3 usage temporal microstructure smoke

This no-submit cell checks a materially different raw source family after sleep/residual closure: coarse mUsageStats temporal microstructure with no direct app-name model columns. Failure is valid closure evidence.


In [ ]:
usage_temporal_path = project_dir / "reports" / "remote_phase3_usage_temporal_microstructure_smoke_20260618.json"
cmd = f"{sys.executable} scripts/diagnose_phase3_usage_temporal_microstructure_smoke_post_id341.py --output {usage_temporal_path}"
run(cmd, cwd=project_dir)

usage_temporal = json.loads(usage_temporal_path.read_text())
print(json.dumps(usage_temporal["decision"], indent=2, ensure_ascii=False))
assert usage_temporal["decision"]["create_submission_artifact"] is False
assert usage_temporal["decision"]["submit_now"] is False
assert usage_temporal["decision"]["push_kaggle_now"] is False
assert usage_temporal["decision"]["run_kaggle_now"] is False
# Do not assert final_usage_temporal_gate_pass: failure closes this source family.


### Usage semantic burst/Markov matrix validation stop

This no-submit cell revalidates the materialized usage semantic burst/Markov feature matrix as label-free infrastructure only. Passing this gate is closure evidence, not permission to train or create ID342.


In [ ]:
usage_semantic_stop_path = project_dir / "reports" / "remote_post_id341_usage_semantic_burst_markov_feature_matrix_validation_stop_20260619.json"
cmd = f"{sys.executable} scripts/diagnose_post_id341_usage_semantic_burst_markov_feature_matrix_validation_stop.py --output {usage_semantic_stop_path}"
run(cmd, cwd=project_dir)

usage_semantic_stop = json.loads(usage_semantic_stop_path.read_text())
print(json.dumps(usage_semantic_stop["decision"], indent=2, ensure_ascii=False))
assert usage_semantic_stop["decision"]["validation_stop_passed"] is True
assert usage_semantic_stop["decision"]["public_058_evidence"] is False
assert usage_semantic_stop["decision"]["score_route_opened"] is False
assert usage_semantic_stop["decision"]["train_model_now"] is False
assert usage_semantic_stop["decision"]["create_submission_artifact"] is False
assert usage_semantic_stop["decision"]["submit_now"] is False
assert usage_semantic_stop["decision"]["push_kaggle_now"] is False
assert usage_semantic_stop["decision"]["run_kaggle_now"] is False


In [ ]:
self_supervised_design_path = project_dir / "reports" / "remote_post_id341_self_supervised_subject_state_embedding_design_20260619.json"
cmd = f"{sys.executable} scripts/diagnose_post_id341_self_supervised_subject_state_embedding_design.py --output {self_supervised_design_path}"
run(cmd, cwd=project_dir)

self_supervised_design = json.loads(self_supervised_design_path.read_text())
print(json.dumps(self_supervised_design["decision"], indent=2, ensure_ascii=False))
assert self_supervised_design["decision"]["design_ready"] is True
assert self_supervised_design["decision"]["source_family"] == "self_supervised_subject_state_embedding_v1"
assert self_supervised_design["decision"]["local_no_approval_allowed"] is False
assert self_supervised_design["decision"]["requires_explicit_heavy_compute_approval"] is True
assert self_supervised_design["decision"]["public_058_evidence"] is False
assert self_supervised_design["decision"]["score_route_opened"] is False
assert self_supervised_design["decision"]["feature_matrix_allowed_now"] is False
assert self_supervised_design["decision"]["train_model_now"] is False
assert self_supervised_design["decision"]["create_submission_artifact"] is False
assert self_supervised_design["decision"]["submit_now"] is False
assert self_supervised_design["decision"]["push_kaggle_now"] is False
assert self_supervised_design["decision"]["run_kaggle_now"] is False


In [ ]:

report_path = project_dir / "reports" / "remote_post_id341_upstream_rebuild_blueprint_20260618.json"
cmd = f"{sys.executable} scripts/diagnose_post_id341_upstream_rebuild_blueprint.py --output-json {report_path}"
run(cmd, cwd=project_dir)

import json
report = json.loads(report_path.read_text())
print(json.dumps(report["decision"], indent=2, ensure_ascii=False))
assert report["decision"]["blueprint_ready"] is True
assert report["decision"]["create_submission_artifact"] is False
assert report["decision"]["submit_now"] is False


### Phase-1 readiness audit

This second report checks whether the packaged project is ready for the report-first upstream rebuild lane without creating any candidate artifact.


In [ ]:

readiness_path = project_dir / "reports" / "remote_post_id341_upstream_rebuild_phase1_readiness_20260618.json"
cmd = f"{sys.executable} scripts/diagnose_post_id341_upstream_rebuild_phase1_readiness.py --output {readiness_path}"
run(cmd, cwd=project_dir)

readiness = json.loads(readiness_path.read_text())
print(json.dumps(readiness["decision"], indent=2, ensure_ascii=False))
assert readiness["decision"]["phase1_ready"] is True
assert readiness["decision"]["create_submission_artifact"] is False
assert readiness["decision"]["submit_now"] is False
assert readiness["decision"]["push_kaggle_now"] is False
assert readiness["decision"].get("local_new_source_stack_closed") is True


### Upstream source priority matrix

This third report ranks the existing upstream source artifacts by frozen split behavior, public-failure geometry, and dependency role so the next heavy run starts from the cleanest useful backbone.


In [ ]:

priority_path = project_dir / "reports" / "remote_post_id341_upstream_source_priority_matrix_20260618.json"
cmd = f"{sys.executable} scripts/diagnose_post_id341_upstream_source_priority_matrix.py --output {priority_path}"
run(cmd, cwd=project_dir)

priority = json.loads(priority_path.read_text())
print(json.dumps(priority["decision"], indent=2, ensure_ascii=False))
assert priority["decision"]["any_existing_artifact_promotable"] is False
assert priority["decision"]["create_submission_artifact"] is False
assert priority["decision"]["submit_now"] is False
assert priority["decision"]["push_kaggle_now"] is False


### Clean ID221/ID226 backbone rebuild design

This fourth report turns the priority matrix into a concrete no-submit contract for the next heavy Kaggle revision.


In [ ]:

design_path = project_dir / "reports" / "remote_post_id341_clean_backbone_rebuild_design_20260618.json"
cmd = (
    f"{sys.executable} scripts/diagnose_post_id341_clean_backbone_rebuild_design.py "
    f"--priority-report {priority_path} --output {design_path}"
)
run(cmd, cwd=project_dir)

design = json.loads(design_path.read_text())
print(json.dumps(design["decision"], indent=2, ensure_ascii=False))
assert design["decision"]["clean_backbone_design_ready"] is True
assert design["decision"]["create_submission_artifact"] is False
assert design["decision"]["submit_now"] is False
assert design["decision"]["push_kaggle_now"] is False
assert design["decision"]["run_heavy_rebuild_now"] is False


### Clean ID221 no-submit probes

These probes check whether the ID221-like feature-similarity source survives fold-internal preprocessing and inner selection before any clean ID226 work is added.


In [ ]:

id221_smoke_path = project_dir / "reports" / "remote_post_id341_clean_id221_feature_similarity_smoke_20260618.json"
cmd = f"{sys.executable} scripts/diagnose_post_id341_clean_id221_feature_similarity_smoke.py --output {id221_smoke_path}"
run(cmd, cwd=project_dir)

id221_smoke = json.loads(id221_smoke_path.read_text())
print(json.dumps(id221_smoke["decision"], indent=2, ensure_ascii=False))
assert id221_smoke["decision"]["create_submission_artifact"] is False
assert id221_smoke["decision"]["submit_now"] is False


In [ ]:

id221_inner_path = project_dir / "reports" / "remote_post_id341_clean_id221_inner_selection_probe_20260618.json"
cmd = f"{sys.executable} scripts/diagnose_post_id341_clean_id221_inner_selection_probe.py --output {id221_inner_path}"
run(cmd, cwd=project_dir)

id221_inner = json.loads(id221_inner_path.read_text())
print(json.dumps(id221_inner["decision"], indent=2, ensure_ascii=False))
assert id221_inner["decision"]["create_submission_artifact"] is False
assert id221_inner["decision"]["submit_now"] is False
assert id221_inner["decision"]["push_kaggle_now"] is False


In [ ]:

next_step_path = project_dir / "reports" / "remote_post_id341_clean_backbone_next_step_decision_20260618.json"
cmd = (
    f"{sys.executable} scripts/diagnose_post_id341_clean_backbone_next_step_decision.py "
    f"--smoke-report {id221_smoke_path} --inner-report {id221_inner_path} "
    f"--design-report {design_path} --output {next_step_path}"
)
run(cmd, cwd=project_dir)

next_step = json.loads(next_step_path.read_text())
print(json.dumps(next_step["decision"], indent=2, ensure_ascii=False))
assert next_step["decision"]["move_to_clean_id226_design_or_smoke"] is True
assert next_step["decision"]["create_submission_artifact"] is False
assert next_step["decision"]["submit_now"] is False


### Clean ID226 calibration smoke and lane closure

These reports test whether the ID226-like calibration layer survives without ID172 full-OOF extreme calibration, then close or continue the clean-backbone lane.


In [ ]:

id226_smoke_path = project_dir / "reports" / "remote_post_id341_clean_id226_calibration_smoke_20260618.json"
cmd = f"{sys.executable} scripts/diagnose_post_id341_clean_id226_calibration_smoke.py --output {id226_smoke_path}"
run(cmd, cwd=project_dir)

id226_smoke = json.loads(id226_smoke_path.read_text())
print(json.dumps(id226_smoke["decision"], indent=2, ensure_ascii=False))
assert id226_smoke["decision"]["create_submission_artifact"] is False
assert id226_smoke["decision"]["submit_now"] is False
assert id226_smoke["decision"]["push_kaggle_now"] is False


In [ ]:

closure_path = project_dir / "reports" / "remote_post_id341_clean_backbone_lane_closure_20260618.json"
cmd = (
    f"{sys.executable} scripts/diagnose_post_id341_clean_backbone_lane_closure.py "
    f"--next-step-report {next_step_path} --id226-report {id226_smoke_path} --output {closure_path}"
)
run(cmd, cwd=project_dir)

closure = json.loads(closure_path.read_text())
print(json.dumps(closure["decision"], indent=2, ensure_ascii=False))
assert closure["decision"]["close_clean_backbone_lane_now"] is True
assert closure["decision"]["create_submission_artifact"] is False
assert closure["decision"]["submit_now"] is False


All reports must remain green before any later notebook revision adds heavier source-model cells. The refreshed closure now blocks three tempting local branches: (1) clean ID221/ID226/ID227/228/237 escalation, (2) the current local new-source stack, and (3) external-prior stacking around ID341, because the best combined sleep/evening/external row still fails to beat the ID178 public anchor. Do not push/run a long Kaggle job without explicit approval.


## 4. Stage report artifacts for Kaggle output

The runner stages only report artifacts and a small manifest under `kaggle_outputs/`. No submission CSV is generated or staged.

In [ ]:

import json
from datetime import datetime, timezone

out_dir = Path(os.environ.get("KAGGLE_OUTPUT_DIR", "kaggle_outputs"))
out_dir.mkdir(parents=True, exist_ok=True)

staged = []
report_roots = [
    transfer_v2_path,
    external_prior_path,
    combined_external_path,
    phase2_spec_path,
    q2q3s3_matrix_path,
    q2q3s3_fallback_chain_path,
    q3s3_negative_evidence_path,
    q3s3_fold_nested_sibling_path,
    q3s3_remaining_route_inventory_path,
    phase2_plan_path,
    block_a_path,
    block_a_gap_path,
    block_b_path,
    usage_temporal_path,
    usage_semantic_stop_path,
    self_supervised_design_path,
    report_path,
    readiness_path,
    priority_path,
    design_path,
    id221_smoke_path,
    id221_inner_path,
    next_step_path,
    id226_smoke_path,
    closure_path,
]
for root_path in report_roots:
    for src in [root_path, root_path.with_suffix(".md")]:
        if src.exists():
            dst = out_dir / src.name
            shutil.copy2(src, dst)
            staged.append(str(dst))

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "runner": "post_id341_phase2_and_clean_backbone_no_submit_runner",
    "project_dir": str(project_dir),
    "staged_files": staged,
    "no_submission_created": True,
    "no_dacon_submission_made": True,
    "local_new_source_stack_closed": readiness["decision"].get("local_new_source_stack_closed"),
    "next_allowed_action": "Run the local output-intake gate on pulled reports; only if it passes may a separate artifact-contract review begin. Do not create or submit a CSV from this runner.",
}
manifest_path = out_dir / "post_id341_upstream_rebuild_runner_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
print(json.dumps(manifest, indent=2, ensure_ascii=False))




## Summary

### Data Analysis Key Findings
- This runner is a no-submit scaffold: it validates latest closure gates, phase-2 raw-source blocks, usage semantic burst/Markov matrix stop-gate, upstream readiness, clean-ID221 probes, clean-ID226 smoke, and clean-backbone closure, then stages the expected remote report artifacts.
- GPU/TPU are disabled by metadata; the current commands create no submission CSV or base prediction artifact.
- Current closure: clean ID221 is anchor/control, clean ID226 calibration does not survive, and ID227/228/237 replay remains blocked.
- The refreshed readiness audit also records that the local post-ID341 new-source stack is closed: sequence+sleep+evening does not beat the ID178 public anchor, and fixed-list place/social novelty selects no target.

### Insights or Next Steps
- If this runner is pushed and pulled reports are green, run the local output-intake gate first; only a separate artifact-contract review can decide whether any later candidate construction is allowed.
- Do not add DACON submission logic to this runner; any future promoted artifact must be created only after pulled reports pass the transfer gate.
